# sync

> Synced navigation between two card stacks — source stack navigation drives target stack to same index

In [ ]:
#| default_exp js.sync

In [ ]:
#| export
from typing import Optional

## Sync JS Generator

Generates a standalone JS snippet that syncs a target card stack to the source
card stack's focused index. When enabled, any navigation in the source stack
triggers `htmx.ajax('POST', targetUrl, ...)` to navigate the target stack to
the same index.

The sync monitors the source stack's hidden `focused_index` input for changes
via a `MutationObserver` on the input's `value` attribute. This catches all
navigation methods (keyboard, scrollbar, scrollwheel, click-to-focus, page jump).

### Listener cleanup

Uses a keyed `window` reference pattern to remove previous observers on re-render,
preventing accumulation across step navigation.

In [ ]:
#| export
def generate_card_stack_sync_js(
    source_input_id:str,  # ID of source stack's focused_index hidden input
    target_nav_url:str,  # URL for target stack's nav_to_index route
    target_total:int,  # Total items in target stack (for bounds check)
    toggle_fn_name:str="toggleSyncedNav",  # Name for window toggle function
    sync_key:str="_cardStackSync",  # Window key for state + cleanup
    target_include_id:str="",  # ID of target stack's focused_index input to include
) -> str:  # Standalone JS snippet (not inside any IIFE)
    """Generate JS for synced navigation between two card stacks.
    
    Source stack navigation drives target stack to the same focused index.
    Toggle on/off via window[toggle_fn_name]() or toolbar button.
    """
    return f"""
    (function() {{
        // Clean up previous sync observer if re-rendered
        if (window.{sync_key} && window.{sync_key}.observer) {{
            window.{sync_key}.observer.disconnect();
        }}

        // Sync state
        var state = window.{sync_key} = {{
            enabled: false,
            targetUrl: '{target_nav_url}',
            maxIndex: {max(0, target_total - 1)},
            observer: null,
            lastSyncedIndex: -1,
        }};

        // Toggle function (called by KeyAction or toolbar button)
        window.{toggle_fn_name} = function() {{
            state.enabled = !state.enabled;
            return state.enabled;
        }};

        // Sync function — navigate target to source's index
        function syncTarget(sourceIndex) {{
            if (!state.enabled) return;
            var idx = parseInt(sourceIndex, 10);
            if (isNaN(idx) || idx < 0 || idx > state.maxIndex) return;
            if (idx === state.lastSyncedIndex) return;
            state.lastSyncedIndex = idx;
            htmx.ajax('POST', state.targetUrl, {{
                values: {{ target_index: idx }},
                swap: 'none',
            }});
        }}

        // Monitor source input for value changes via MutationObserver
        // The hidden input's value is updated by OOB swaps after navigation
        var sourceInput = document.getElementById('{source_input_id}');
        if (sourceInput) {{
            // Use afterSettle to detect when source stack navigation completes
            var settleKey = '{sync_key}_settle';
            if (window[settleKey]) {{
                document.body.removeEventListener('htmx:afterSettle', window[settleKey]);
            }}
            var settleHandler = function(evt) {{
                if (!state.enabled) return;
                var input = document.getElementById('{source_input_id}');
                if (input) syncTarget(input.value);
            }};
            window[settleKey] = settleHandler;
            document.body.addEventListener('htmx:afterSettle', settleHandler);
        }}
    }})();
    """

## Tests

In [ ]:
# Test basic generation
js = generate_card_stack_sync_js(
    source_input_id="seg-focused-index",
    target_nav_url="/align/nav_to_index",
    target_total=15,
    toggle_fn_name="toggleSyncedNav",
    sync_key="_segAlignSync",
)

# Verify key elements are present
assert 'toggleSyncedNav' in js
assert '_segAlignSync' in js
assert 'seg-focused-index' in js
assert '/align/nav_to_index' in js
assert 'maxIndex: 14' in js  # 15 - 1
assert 'htmx.ajax' in js
assert 'state.enabled' in js
assert 'observer.disconnect' in js  # Cleanup
assert 'afterSettle' in js
assert 'lastSyncedIndex' in js  # Dedup guard

# Test zero items edge case
js_empty = generate_card_stack_sync_js(
    source_input_id="src",
    target_nav_url="/tgt/nav",
    target_total=0,
)
assert 'maxIndex: 0' in js_empty

print('Sync JS generator tests passed')

Sync JS generator tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()